In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

print("Installing required packages...")

!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-groq \
langchain-core \
faiss-cpu \
pypdf \
sentence-transformers \
streamlit \
langsmith

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, whic

In [2]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

LANGCHAIN_API_KEY = user_secrets.get_secret("LANGCHAIN_API_KEY")
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

print("LangSmith Key Loaded:", LANGCHAIN_API_KEY[:10], "...")
print("Groq Key Loaded:", GROQ_API_KEY[:10], "...")

LangSmith Key Loaded: lsv2_pt_4d ...
Groq Key Loaded: gsk_L8t9Kg ...


In [3]:
import os
from kaggle_secrets import UserSecretsClient
from cryptography.fernet import Fernet

user_secrets = UserSecretsClient()

os.environ["LANGCHAIN_API_KEY"] = user_secrets.get_secret("LANGCHAIN_API_KEY")
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "zyro-rag-challenge"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully!")

Environment configured successfully!


In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Folder containing all HR policy PDFs
pdf_folder = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

documents = []

# Load every PDF
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages from all PDFs.")

/tmp/ipykernel_16/4117267071.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 39 pages from all PDFs.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

# Split the documents into chunks
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")

Created 168 chunks.


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

print("Embedding model initialized successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized successfully!


In [7]:
from langchain_community.vectorstores import FAISS

# Build the FAISS vector database
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Create an MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.8
    }
)

print("✅ Vector Store created successfully!")
print("Total vectors:", vectorstore.index.ntotal)

✅ Vector Store created successfully!
Total vectors: 168


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=300
)

print("✅ Groq LLM initialized successfully!")

✅ Groq LLM initialized successfully!


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt Template
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are the official HR Assistant for Zyro Dynamics.

Answer ONLY using the provided HR policy context.

Rules:
1. Give a complete and accurate answer.
2. Include all relevant numbers, dates, durations, eligibility, limits, and policy names exactly as provided.
3. If information is spread across multiple sections, combine it into one answer.
4. Do not add information that is not in the context.
5. If the question cannot be answered from the HR policy context, reply exactly:
"I can only answer HR-related questions based on the Zyro Dynamics HR policy documents."

Context:
{context}

Question:
{question}

Answer:
""")

# Convert retrieved documents into one string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Main RAG function
def rag_chain(question: str):
    docs = retriever.invoke(question)

    context = format_docs(docs)

    response = (
        RAG_PROMPT
        | llm
        | StrOutputParser()
    ).invoke({
        "context": context,
        "question": question
    })

    return {
        "answer": response,
        "sources": docs
    }

print("✅ RAG Chain created successfully!")

✅ RAG Chain created successfully!


In [10]:
question = "How many earned leaves are employees entitled to?"

result = rag_chain(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(result["answer"])

print("\nSOURCE DOCUMENTS:")
for i, doc in enumerate(result["sources"], 1):
    print(f"\nSource {i}:")
    print(doc.metadata)

QUESTION:
How many earned leaves are employees entitled to?

ANSWER:
Employees become eligible for 15 days of Earned Leave upon completion of one year of continuous service, provided they have worked for a minimum of 240 days.

SOURCE DOCUMENTS:

Source 1:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T10:58:03+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-20T10:58:03+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/02_Leave_Policy.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}

Source 2:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T10:58:03+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-20T10:58:03+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source':

In [11]:
REFUSAL_MESSAGE = (
    "I can only answer HR-related questions based on the Zyro Dynamics HR policy documents."
)

def ask_bot(question: str):
    docs = retriever.invoke(question)

    # If no relevant documents found
    if len(docs) == 0:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": []
        }

    context = format_docs(docs)

    # Simple relevance check
    if len(context.strip()) < 50:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": []
        }

    response = (
        RAG_PROMPT
        | llm
        | StrOutputParser()
    ).invoke({
        "context": context,
        "question": question
    })

    return {
        "answer": response,
        "sources": list(set(
            os.path.basename(doc.metadata["source"])
            for doc in docs
        ))
    }

print("✅ Guardrails added successfully!")

✅ Guardrails added successfully!


In [12]:
test_questions = [
    "How many earned leaves are employees entitled to?",
    "Can I work from home permanently?",
    "Who won the FIFA World Cup?"
]

for i, q in enumerate(test_questions, 1):
    print("=" * 70)
    print(f"Question {i}: {q}")

    result = ask_bot(q)

    print("\nAnswer:")
    print(result["answer"])

Question 1: How many earned leaves are employees entitled to?

Answer:
Employees become eligible for 15 days of Earned Leave upon completion of one year of continuous service, provided they have worked for a minimum of 240 days.
Question 2: Can I work from home permanently?

Answer:
I can only answer HR-related questions based on the Zyro Dynamics HR policy documents.
Question 3: Who won the FIFA World Cup?

Answer:
I can only answer HR-related questions based on the Zyro Dynamics HR policy documents.


In [13]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")


HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!



In [14]:
app_code = r'''
import streamlit as st
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------
# Streamlit Page
# ---------------------------

st.set_page_config(
    page_title="Zyro Dynamics HR Assistant",
    page_icon="🤖"
)

st.title("🤖 Zyro Dynamics HR Assistant")
st.write("Ask any question about Zyro Dynamics HR policies.")

# ---------------------------
# Load API Key
# ---------------------------

groq_key = st.secrets["GROQ_API_KEY"]

llm = ChatGroq(
    groq_api_key=groq_key,
    model="llama-3.1-8b-instant",
    temperature=0
)

# ---------------------------
# Load Documents
# ---------------------------

@st.cache_resource
def load_vectorstore():

    pdf_folder = "docs"

    documents = []

    for file in os.listdir(pdf_folder):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(pdf_folder, file))
            documents.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]
    )

    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5"
    )

    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    return vectorstore


vectorstore = load_vectorstore()

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":10
    }
)

# ---------------------------
# Prompt
# ---------------------------

prompt = ChatPromptTemplate.from_template("""
You are the official HR Assistant for Zyro Dynamics.

Answer ONLY using the provided HR policy context.

Rules:
1. Give a complete and accurate answer.
2. Include all relevant numbers, dates, durations, eligibility, limits, and policy names exactly as provided.
3. If information is spread across multiple sections, combine it into one answer.
4. Do not add information that is not in the context.
5. If the question cannot be answered from the HR policy context, reply exactly:
"I can only answer HR-related questions based on the Zyro Dynamics HR policy documents."

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def ask_bot(question):

    docs = retriever.invoke(question)

    context = format_docs(docs)

    response = (
        prompt
        | llm
        | StrOutputParser()
    ).invoke({
        "context": context,
        "question": question
    })

    sources = sorted(
        list(
            set(
                os.path.basename(doc.metadata["source"])
                for doc in docs
            )
        )
    )

    return response, sources

# ---------------------------
# Chat Interface
# ---------------------------

question = st.text_input(
    "Ask your HR question:",
    placeholder="Example: How many earned leaves are employees entitled to?"
)

if st.button("Get Answer"):

    if question.strip() == "":
        st.warning("Please enter a question.")

    else:
        with st.spinner("Searching HR policies..."):

            answer, sources = ask_bot(question)

        st.subheader("Answer")
        st.write(answer)

        st.subheader("Source Documents")

        for src in sources:
            st.write(f"• {src}")

'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created successfully!")

✅ app.py created successfully!


In [15]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),    
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ==")]
     
eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

15 evaluation questions loaded.


In [16]:
import re
import time
import csv

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

streamlit_link = "https://zyro-rag-sanju.streamlit.app/"

langsmith_link = "https://smith.langchain.com/public/8cbc5156-d073-436d-9df1-452fce9d77b1/r/019f0ad7-0247-7652-ad1c-8b319ff97d7c"

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    raise ValueError("Please correct the links and re-run the cell.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator

Generating responses for 15 questions...

[01/15] Q01 ... OK
[02/15] Q02 ... OK
[03/15] Q03 ... OK
[04/15] Q04 ... OK
[05/15] Q05 ... OK
[06/15] Q06 ... OK
[07/15] Q07 ... OK
[08/15] Q08 ... OK
[09/15] Q09 ... OK
[10/15] Q10 ... OK
[11/15] Q11 ... OK
[12/15] Q12 ... OK
[13/15] Q13 ... OK
[14/15] Q14 ... OK
[15/15] Q15 ... OK

submission.csv generated successfully.


In [17]:
import re
import csv
import os

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open("submission.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(r["streamlit_link"].strip())
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(r["langsmith_link"].strip())
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run the previous cell first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")

Final Submission Check
submission.csv found (15 rows)
Required columns present: True
Streamlit links valid: True
LangSmith links valid: True
Upload submission.csv to Kaggle to complete your submission.


In [18]:
for i, q in enumerate(eval_questions):
    print(f"\nQuestion {i+1}/{len(eval_questions)}")
    print(q["question"])

    try:
        result = ask_bot(q["question"])
        print("✅ Success")
        print(result["answer"])
    except Exception as e:
        print("❌ ERROR:", e)
        break


Question 1/15
At what rate does Earned Leave accrue per month at Acrux Dynamics, and how many days are employees entitled to after completing one year of service?
✅ Success
According to the Zyro Dynamics HR policy documents, employees are entitled to 15 days of Earned Leave upon completion of one year of continuous service. 

Earned Leave accrues at the rate of 1.25 days per month for employees who have completed one year of continuous service.

Question 2/15
What is the maximum number of Earned Leave days that can be carried forward at the end of the financial year? What happens to the excess balance?
✅ Success
According to the Zyro Dynamics Leave Policy (ZDL-HR-002), a maximum of 45 days of Earned Leave may be carried forward at the end of each financial year (31 March). Any balance exceeding this limit will be automatically encashed at the employee's basic daily rate and credited in the April payroll.

Question 3/15
How many weeks of maternity leave is an employee entitled to, and 

In [19]:
requirements = """streamlit
langchain
langchain-community
langchain-text-splitters
langchain-huggingface
langchain-groq
langchain-core
langsmith
faiss-cpu
pypdf
sentence-transformers
transformers
torch
huggingface_hub
groq
python-dotenv
tiktoken
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt created successfully!")

✅ requirements.txt created successfully!
